In [27]:
import re
from collections import Counter
from pathlib import Path
from typing import List

# Monorepo root: walk up from cwd until we find pyproject.toml, then use its data/
_p = Path.cwd().resolve()
DATA_DIR = None
while _p != _p.parent:
    if (_p / "pyproject.toml").is_file():
        DATA_DIR = _p / "data"
        break
    _p = _p.parent
if DATA_DIR is None:
    DATA_DIR = Path.cwd().resolve() / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)

In [28]:
# Run if not data
data = [
    "Ths is a smple txt with errrs.",
    "This is a clean sentence.",
    "0CR scann3d d0cum3nt with nois3",
    "lI l IlI random garbage xqztr",
    "The quick brown fox jumps over the lazy dog.",
    "Eman, as carrier, walks towards the graveside, the other Eman having gone."
]

## Load raw scraped text lines

This cell aggregates the raw scraped corpus from `DATA_DIR` by:

- Reading every `*.txt` file in the folder (UTF-8)
- Splitting into individual lines
- Stripping whitespace and dropping empty lines
- Returning a single flat `list[str]` of lines


In [29]:
from pathlib import Path

def load_txt_lines(data_dir: Path):
    lines = []
    for path in sorted(data_dir.glob("*.txt")):
        try:
            raw = path.read_text(encoding="utf-8")
            for line in raw.splitlines():
                s = line.strip()
                if s:
                    lines.append(s)
        except OSError as e:
            print(f"  Warning: could not read {path}: {e}")
    return lines


txt_paths = sorted(DATA_DIR.glob("*.txt"))
data = load_txt_lines(DATA_DIR)

print(f"{len(txt_paths)} .txt files found with {len(data)} lines")


160 .txt files found with 100964 lines


## Split lines into sentence chunks

This cell converts the raw line-level corpus (`data`) into a sentence-level corpus by:

- Splitting each line on sentence boundaries (`.`, `!`, `?` followed by whitespace)
- Appending each resulting sentence into a single flat list (`texts`)

The final print shows the **total number of sentence chunks** produced.

In [30]:
import re
from typing import List
import random


def split_sentences(text: str) -> List[str]:
    """One sentence per item; splits on . ! ? followed by whitespace."""
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]


data_slice = data
texts = []

for line_idx, line in enumerate(data_slice):
    sentences = split_sentences(line)
    for sentence in sentences:
        texts.append(sentence)

    # print(f"For line-{line_idx}, we got {len(sentences)} sentences")

print(f"New total count of lines found after split into chunks = {len(texts)}")


New total count of lines found after split into chunks = 751852


## Layer 1: Sanity filtering

This cell applies lightweight quality filters to remove obvious OCR noise before heavier processing.

It keeps chunks only when they pass both checks:

- **Mostly alphabetic text**: at least 70% of characters are letters
- **Reasonable average word length**: average token length is between 2 and 12 characters


In [ ]:
import random

def is_mostly_text(chunk, threshold=0.7):
    letters = sum(c.isalpha() for c in chunk)
    return letters / max(len(chunk), 1) >= threshold

def has_reasonable_word_lengths(chunk, min_avg=2, max_avg=12):
    words = chunk.split()
    if not words:
        return False
    avg_len = sum(len(w) for w in words) / len(words)
    return min_avg <= avg_len <= max_avg

def passes_basic_checks(chunk):
    return (
        is_mostly_text(chunk) and
        has_reasonable_word_lengths(chunk)
    )

filtered_basic = [c for c in texts if passes_basic_checks(c)]

# Avoid printing the full list: with ~750k+ chunks it can hang the UI or OOM.
print(f"After Layer 1: kept {len(filtered_basic)} / {len(texts)} chunks")

sample = random.sample(filtered_basic, min(5, len(filtered_basic)))
print("Sample (random 5):", sample)
print("Sample (first 5, randomized):", filtered_basic)



## Layer 2: Language check (English)

This cell filters the dataset to primarily English text using the `lingua` library, which is robust and works offline.

Key aspects:

-   **`_language_detector()`**: Lazily initialized (cached) to detect all spoken languages, improving accuracy for non-English identification.
-   **`passes_english_language_check()`**: Keeps chunks detected as English.
    -   Very short chunks (< 20 characters) are **skipped** for detection (assumed good if they passed Layer 1), as `lingua` is less reliable on extremely short text.


In [ ]:
# Uses lingua (accurate on noisy text; works fully offline).
import random
from functools import lru_cache
from lingua import Language, LanguageDetectorBuilder



@lru_cache(maxsize=1)
def _language_detector():
    # Full spoken-language set so non-English lines get a competing hypothesis
    return LanguageDetectorBuilder.from_all_spoken_languages().build()


def passes_english_language_check(text, min_chars_for_id=20):
    """
    Keep chunks that are detected as English. Very short snippets are skipped
    (unreliable); they are kept if they passed Layer 1.
    """
    t = text.strip()
    if len(t) < min_chars_for_id:
        return True
    lang = _language_detector().detect_language_of(t)
    return lang == Language.ENGLISH


filtered_english = [c for c in filtered_basic if passes_english_language_check(c)]

print(f"After English filter: kept {len(filtered_english)} / {len(filtered_basic)} filtered_basic")

sample = random.sample(filtered_english, min(5, len(filtered_english)))
print("Sample (random 5):", sample)
print("Sample (first 5, randomized):", filtered_english)

## Layer 3: Final decision logic

This cell applies the final set of filtering rules to the dataset to ensure high quality.

Key functions:

-   **`english_ratio(text)`**: A heuristic that calculates the fraction of alphabetic characters in a text. This provides a simple measure of text quality without relying on heavier language models.
-   **`score_text(text)`**: Combines the `english_ratio` and `length` of the text into a single dictionary for easier evaluation.
-   **`final_filter(text)`**: Applies the core filtering logic:
    -   Rejects chunks where the `english_ratio` is less than 0.6 (i.e., less than 60% alphabetic characters).
    -   Rejects chunks shorter than 10 characters.

In [ ]:
import random

# Layer 3 - Final decision logic
def english_ratio(text: str) -> float:
    """Heuristic: fraction of characters that are alphabetic.

    This keeps `final_filter()` independent of any heavy language models.
    """
    letters = sum(c.isalpha() for c in text)
    return letters / max(len(text), 1)


def score_text(text):
    return {
        "english_ratio": english_ratio(text),
        "length": len(text)
    }

def final_filter(text):
    score = score_text(text)
    
    if score["english_ratio"] < 0.6:
        return False
    if score["length"] < 10:
        return False
    
    return True

final_dataset = [c for c in filtered_english if final_filter(c)]

# print("Final dataset:", final_dataset)

print(f"Final dataset: {len(final_dataset)} / {len(filtered_english)} chunks")
sample = random.sample(filtered_english, min(5, len(filtered_english)))
print("Sample (random 5):", sample)

Final dataset: 4 / 4 chunks
Sample (random 5): ['This is a clean sentence.', 'Eman, as carrier, walks towards the graveside, the other Eman having gone.']


## Save the refined dataset

This cell is responsible for saving the `final_dataset` (processed in Layer 3) into two formats:

-   **Plain text file (`refined_corpus.txt`)**: Each chunk is written on a new line.
-   **JSON array (`refined_corpus.json`)**: The entire dataset is saved as a JSON array.

The output files are stored in a `refined` subdirectory within the `pdf_scrape` folder. The cell also prints confirmation messages showing the number of chunks written and the paths to the saved files.

In [ ]:
# Save the refined dataset (run after Layer 4 defines `final_dataset`).
import json

_refined_root = DATA_DIR.parent / "pdf_scrape" / "refined"
_refined_root.mkdir(parents=True, exist_ok=True)

_txt_out = _refined_root / "refined_corpus.txt"
_json_out = _refined_root / "refined_corpus.json"

_txt_out.write_text("\n".join(final_dataset) + "\n", encoding="utf-8")

with _json_out.open("w", encoding="utf-8") as f:
    json.dump(final_dataset, f, ensure_ascii=False)

print(f"Wrote {len(final_dataset):,} chunks to {_txt_out}")
print(f"Wrote {len(final_dataset):,} chunks to {_json_out}")

Wrote 663,678 chunks to /Users/roqqu/Desktop/Build An LLM/build-an-llm/pdf_scrape/refined/refined_corpus.txt
Wrote 663,678 chunks to /Users/roqqu/Desktop/Build An LLM/build-an-llm/pdf_scrape/refined/refined_corpus.json
